# morphological_quantification_2026-01-02 — 00_manifest_qc

**Feeds:** Fig 3f

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# 00 | Manifest And Metadata QC

Build the raw CZI manifest, initialize the editable condition-label template, and sanity-check acquisition consistency before any downstream morphology quantification.

## Cell Guide

- `Setup`: resolve the project root, import helper code, and define output paths.
- `Build Raw Manifest`: scan the CZI files, extract metadata, and write canonical manifest outputs.
- `Manual Condition Template`: create or refresh the editable file-to-condition mapping template.
- `Manifest QC`: review acquisition dates, shapes, channels, and flagged batches.
- `Representative Previews`: spot-check a small subset of images directly from the raw CZI files.
- `Next Step`: move into `01`, then `02`, then `03` for per-z outlining, consensus geometry, and manual posterior annotation.

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

if (Path.cwd() / "scripts").exists():
    ROOT = Path.cwd()
elif (Path.cwd().parent / "scripts").exists():
    ROOT = Path.cwd().parent
else:
    raise RuntimeError("Could not resolve project root from current working directory.")

MPLCONFIGDIR = ROOT / "results" / "tmp" / "mplconfig"
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts import morphology_quantification_helpers as mqh
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", 100)
ROOT

## Settings Notes

- This notebook does **not** guess condition labels from filenames. It writes a manual template and expects us to fill that mapping honestly.
- Files with descriptive suffixes, duplicate organoid IDs, or outlier acquisition dates stay in the manifest by default, but they are flagged for extra review.
- Raw data stay in `data/`; everything generated here is written under `results/manifests/`.

In [ ]:
COHORT_ID = "2026-01-02_day5_pax8"
DATA_ROOT = ROOT / "data" / "day5 after fix and stain for PAX8-647"
RESULTS_DIR = ROOT / "results"
MANIFEST_DIR = RESULTS_DIR / "manifests"
RAW_MANIFEST_PATH = MANIFEST_DIR / "raw_czi_manifest.tsv"
MANUAL_CONDITION_PATH = MANIFEST_DIR / "manual_condition_labels.tsv"
ANALYSIS_MANIFEST_PATH = MANIFEST_DIR / "analysis_manifest.tsv"
PREVIEW_COUNT = 6

for path in [RESULTS_DIR, MANIFEST_DIR, RESULTS_DIR / "tmp"]:
    path.mkdir(parents=True, exist_ok=True)

RAW_MANIFEST_PATH, MANUAL_CONDITION_PATH, ANALYSIS_MANIFEST_PATH

## Build Raw Manifest

In [ ]:
manifest_df = mqh.build_raw_czi_manifest(
    data_root=DATA_ROOT,
    project_root=ROOT,
    cohort_id=COHORT_ID,
)
manual_condition_df = mqh.sync_manual_condition_template(
    manifest_df=manifest_df,
    template_path=MANUAL_CONDITION_PATH,
)
analysis_manifest_df = mqh.build_analysis_manifest(
    manifest_df=manifest_df,
    manual_condition_df=manual_condition_df,
)

manifest_df.to_csv(RAW_MANIFEST_PATH, sep="\t", index=False)
analysis_manifest_df.to_csv(ANALYSIS_MANIFEST_PATH, sep="\t", index=False)

print(f"Wrote raw manifest: {RAW_MANIFEST_PATH}")
print(f"Wrote/updated manual condition template: {MANUAL_CONDITION_PATH}")
print(f"Wrote analysis manifest: {ANALYSIS_MANIFEST_PATH}")
print(f"Images discovered: {len(manifest_df)}")

display(analysis_manifest_df.head())

## Manual Condition Template

In [ ]:
missing_condition_df = analysis_manifest_df.loc[
    analysis_manifest_df["condition_label"].fillna("").astype(str).str.strip().eq(""),
    [
        "file_id",
        "canonical_position",
        "acquisition_date",
        "condition_label",
        "condition_group",
        "notes",
        "file_path",
    ],
].copy()

print("Fill the editable mapping here before group-level stats:")
print(MANUAL_CONDITION_PATH)
print()
print(f"Rows still missing condition_label: {len(missing_condition_df)}")

display(missing_condition_df.head(12))

## Manifest QC

In [ ]:
date_summary_df = (
    analysis_manifest_df.groupby("acquisition_date", dropna=False)
    .agg(
        n_images=("file_path", "size"),
        n_flagged=("batch_warning_flag", "sum"),
    )
    .reset_index()
    .sort_values("acquisition_date")
)
channel_summary_df = (
    analysis_manifest_df.groupby("channel_names", dropna=False)
    .size()
    .rename("n_images")
    .reset_index()
    .sort_values("n_images", ascending=False)
)
shape_summary_df = analysis_manifest_df[["size_x", "size_y", "size_z"]].describe().T
review_df = analysis_manifest_df.loc[
    analysis_manifest_df["batch_warning_flag"] | analysis_manifest_df["name_review_flag"],
    [
        "file_id",
        "canonical_position",
        "file_stem",
        "filename_suffix",
        "acquisition_date",
        "batch_warning_flag",
        "batch_warning_reason",
        "name_review_flag",
        "name_review_note",
        "file_path",
    ],
].copy()

display(date_summary_df)
display(channel_summary_df)
display(shape_summary_df)

if len(review_df):
    print("Flagged subset for extra review:")
    display(review_df)

## Representative Previews

In [ ]:
preview_df = analysis_manifest_df.head(min(PREVIEW_COUNT, len(analysis_manifest_df))).copy()
fig, axes = plt.subplots(2, len(preview_df), figsize=(3.0 * len(preview_df), 6), constrained_layout=True)
if len(preview_df) == 1:
    axes = axes.reshape(2, 1)

for col, row in enumerate(preview_df.itertuples(index=False)):
    projected = mqh.load_projected_channels(ROOT / row.file_path, projection="max")
    axes[0, col].imshow(mqh.robust_rescale(projected["dapi"]), cmap="gray")
    axes[0, col].set_title(f"{row.file_id} | DAPI")
    axes[1, col].imshow(projected["overlay_rgb"])
    axes[1, col].set_title(f"{row.file_id} | overlay")
    for axis in [axes[0, col], axes[1, col]]:
        axis.set_xticks([])
        axis.set_yticks([])

plt.show()

## Next Step

Move into `01_per_z_whole_morph_candidates.ipynb` for per-z DAPI outlining, `02_whole_morph_geometry.ipynb` for consensus-axis review, and `03_posterior_annotation.ipynb` for the manual posterior-near click.